# **Generating Samples Notebook**

## Locate Our Model Checkpoints

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
import h5py

In [2]:
PROJECT_ROOT = Path("/home/muhamed/repo/CTDDG")
REWORKED_ROOT = PROJECT_ROOT / "REWORKED_CODE"

EVALUATION_ROOT = REWORKED_ROOT / "EVALUATION_OUTPUT"
EMBEDDINGS_ROOT = EVALUATION_ROOT / "embeddings"
SAMPLES_ROOT = EVALUATION_ROOT / "generated_samples"

TEST_FOLDS = {
    "1002_103": 103,
    "1042_104": 99,
}

print("=" * 65)
print("NOTEBOOK 2 — MODEL GENERATION SETUP")
print("=" * 65)

print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)


NOTEBOOK 2 — MODEL GENERATION SETUP
Python: /home/muhamed/.conda/envs/ctddg_torch/bin/python
Project: /home/muhamed/repo/CTDDG


In [ ]:
print("\nPROTEIN EMBEDDINGS")

for fold, expected_count in TEST_FOLDS.items():

    folder = EMBEDDINGS_ROOT / fold
    manifest_path = folder / "protein_manifest.csv"
    embedding_path = folder / "prose_embeddings_avg_6165.h5"

    assert manifest_path.is_file(), manifest_path
    assert embedding_path.is_file(), embedding_path

    manifest = pd.read_csv(manifest_path)

    assert len(manifest) == expected_count

    with h5py.File(embedding_path, "r") as h5:
        assert len(h5.keys()) == expected_count
        assert h5["p1"].shape == (6165,)

    print(f"  PASS: {fold} — {expected_count} proteins")


PROTEIN EMBEDDINGS
  PASS: 1002_103 — 103 proteins
  PASS: 1042_104 — 99 proteins


In [4]:
SEARCH_ROOTS = [
    REWORKED_ROOT / "model_weights",
    REWORKED_ROOT / "PRETRAINING_OUTPUT",
    REWORKED_ROOT / "pretraining",
    REWORKED_ROOT / "finetuning",
    REWORKED_ROOT / "testing",
    PROJECT_ROOT / "Results",
]

CHECKPOINT_EXTENSIONS = {
    ".pt", ".pth", ".ckpt", ".params"
}

SKIP_DIRS = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "datasets", "DATA", "PREPROCESSING_OUTPUT",
    "EVALUATION_OUTPUT", "generated_samples",
    "samples", "venv", ".venv", "site-packages",
}

checkpoint_paths = set()

for root in SEARCH_ROOTS:

    if not root.is_dir():
        continue

    for current_dir, dirs, files in os.walk(root):

        relative_depth = len(
            Path(current_dir).relative_to(root).parts
        )

        if relative_depth >= 6:
            dirs[:] = []
        else:
            dirs[:] = [
                name for name in dirs
                if name not in SKIP_DIRS
            ]

        for filename in files:

            path = Path(current_dir) / filename

            if path.suffix.lower() in CHECKPOINT_EXTENSIONS:
                checkpoint_paths.add(path)

checkpoint_paths = sorted(checkpoint_paths)

print("\n" + "=" * 65)
print("AVAILABLE MODEL CHECKPOINTS")
print("=" * 65)

print("Total checkpoint files found:", len(checkpoint_paths))

for index, path in enumerate(checkpoint_paths[:50], 1):

    size_mb = path.stat().st_size / (1024 ** 2)

    print(f"\n{index}. {path}")
    print(f"   Size: {size_mb:.2f} MB")

if len(checkpoint_paths) > 50:
    print(
        f"\n... {len(checkpoint_paths) - 50} "
        "additional checkpoint files not displayed"
    )



AVAILABLE MODEL CHECKPOINTS
Total checkpoint files found: 31

1. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/best.pt
   Size: 79.93 MB

2. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/latest.pt
   Size: 79.93 MB

3. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/block12_smoke_lambda_1/checkpoint.pt
   Size: 79.93 MB

4. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/block12_smoke_lambda_1/model_best.pt
   Size: 26.66 MB

5. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/checkpoint.pt
   Size: 79.93 MB

6. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/model_best.pt
   Size: 26.66 MB

7. /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_la

In [5]:
REFERENCE_NOTEBOOK = (
    REWORKED_ROOT
    / "testing/CTDDG_Five_Model_Generation.ipynb"
)

print("\n" + "=" * 65)
print("GENERATION CODE")
print("=" * 65)

print("Reference notebook:", REFERENCE_NOTEBOOK)
print("Exists:", REFERENCE_NOTEBOOK.is_file())

print("\nBLOCK 1 COMPLETE")
print("No checkpoints loaded.")
print("No molecules generated.")
print("No files modified.")


GENERATION CODE
Reference notebook: /home/muhamed/repo/CTDDG/REWORKED_CODE/testing/CTDDG_Five_Model_Generation.ipynb
Exists: True

BLOCK 1 COMPLETE
No checkpoints loaded.
No molecules generated.
No files modified.


## Identify the Five Models

In [6]:

# ============================================================
# BLOCK 2.1 — INSPECT FIVE-MODEL GENERATION NOTEBOOK
# ============================================================

import json
import re
from pathlib import Path

NOTEBOOK_PATH = (
    REWORKED_ROOT
    / "testing/CTDDG_Five_Model_Generation.ipynb"
)

assert NOTEBOOK_PATH.is_file(), NOTEBOOK_PATH

with NOTEBOOK_PATH.open("r", encoding="utf-8") as f:
    reference_notebook = json.load(f)

print("=" * 65)
print("MODEL CONFIGURATION DISCOVERY")
print("=" * 65)

print("Notebook:", NOTEBOOK_PATH)
print("Cells:", len(reference_notebook["cells"]))

# Search for model names, checkpoints and configurations.
PATTERN = re.compile(
    r"checkpoint|ckpt|pretrain_run|finetun|"
    r"nitesh|model_specs|model_config|"
    r"model_paths|model_entries|n_c\s*=|"
    r"6165|model.*path|variant",
    re.IGNORECASE
)

configuration_matches = []

for cell_index, cell in enumerate(
    reference_notebook["cells"], start=1
):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    code = "".join(source) if isinstance(source, list) else source

    for line_number, line in enumerate(
        code.splitlines(), start=1
    ):

        if PATTERN.search(line):

            configuration_matches.append({
                "cell": cell_index,
                "line": line_number,
                "code": line.strip()[:200],
            })

print("\nMatching configuration lines:",
      len(configuration_matches))

print("\nRelevant definitions:\n")

for item in configuration_matches[:65]:

    print(
        f"Cell {item['cell']:>3} | "
        f"Line {item['line']:>3} | "
        f"{item['code']}"
    )

if len(configuration_matches) > 65:
    print(
        f"\n... {len(configuration_matches)-65} "
        "additional matching lines."
    )

print("\nBLOCK 2.1 COMPLETE")


MODEL CONFIGURATION DISCOVERY
Notebook: /home/muhamed/repo/CTDDG/REWORKED_CODE/testing/CTDDG_Five_Model_Generation.ipynb
Cells: 24

Matching configuration lines: 68

Relevant definitions:

Cell   3 | Line  12 | # Existing pretrained runs. New model checkpoints are read directly from the
Cell   3 | Line  16 | FINETUNE_RUN_DIR = (REWORKED / 'finetuning/finetuning_runs'
Cell   3 | Line  21 | # The historical MXNet nitesh_model is inspected but not loaded by PyTorch.
Cell   3 | Line  23 | 'nitesh_model',
Cell   3 | Line  27 | 'Finetuned_BindingDB_ctddg',
Cell   3 | Line  35 | MODEL_ROOT/'Pretrained_no_toxicity/checkpoint.pt'],
Cell   3 | Line  41 | 'candidates': [MODEL_ROOT/'Pretrained_with_toxicity/checkpoint.pt'],
Cell   3 | Line  52 | 'Finetuned_BindingDB_ctddg': {
Cell   3 | Line  54 | 'candidates': [FINETUNE_RUN_DIR/'ckpt_best.pt',
Cell   3 | Line  55 | FINETUNE_RUN_DIR/'ckpt_last.pt'],
Cell   3 | Line  56 | 'metadata_dir': FINETUNE_RUN_DIR,
Cell   3 | Line  57 | 'dense_relu': False, 

In [7]:

# ============================================================
# BLOCK 2.2 — CHECKPOINT INVENTORY
# ============================================================

import pandas as pd

assert "checkpoint_paths" in globals(), (
    "Run Notebook 2 — Block 1 first."
)

checkpoint_rows = []

for path in checkpoint_paths:

    path = Path(path)

    checkpoint_rows.append({
        "filename": path.name,
        "size_MB": round(
            path.stat().st_size / (1024**2), 2
        ),
        "path": str(path),
    })

checkpoint_df = pd.DataFrame(
    checkpoint_rows,
    columns=["filename", "size_MB", "path"]
)

print("=" * 65)
print("AVAILABLE CHECKPOINT FILES")
print("=" * 65)

print("Total checkpoints:", len(checkpoint_df))

if checkpoint_df.empty:
    print("No checkpoints found in Block 1 search paths.")
else:
    pd.set_option("display.max_colwidth", 150)
    pd.set_option("display.max_rows", 100)

    display(checkpoint_df)

print("\nBLOCK 2.2 COMPLETE")


AVAILABLE CHECKPOINT FILES
Total checkpoints: 31


,filename,size_MB,path
0,best.pt,79.93,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/best.pt
1,latest.pt,79.93,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/latest.pt
2,checkpoint.pt,79.93,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/block12_smoke_lambda_1/checkpoint.pt
3,model_best.pt,26.66,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/block12_smoke_lambda_1/model_best.pt
4,checkpoint.pt,79.93,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/checkpoint.pt
5,model_best.pt,26.66,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/model_best.pt
6,model_final.pt,26.66,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/model_final.pt
7,checkpoint.pt,79.93,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/tune_lambda_0p25/checkpoint.pt
8,model_best.pt,26.66,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/tune_lambda_0p25/model_best.pt
9,checkpoint.pt,79.93,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/tune_lambda_0p50/checkpoint.pt



BLOCK 2.2 COMPLETE


In [8]:

# ============================================================
# BLOCK 2.3 — INSPECT CHECKPOINT METADATA
# ============================================================

import json

print("=" * 65)
print("CHECKPOINT METADATA INSPECTION")
print("=" * 65)

METADATA_FILENAMES = [
    "run_meta.json",
    "metadata.json",
    "config.json",
    "provenance.json",
    "training_config.json",
    "best_checkpoint_meta.json",
]

INTERESTING_KEYS = {
    "model_config",
    "architecture",
    "variant",
    "pretrain_variant",
    "objective",
    "n_c",
    "embedding_dim",
    "protein_embedding_dim",
    "epoch",
    "step",
    "global_step",
    "best_metric",
    "checkpoint",
}

inspected = set()
metadata_found = 0

for checkpoint in checkpoint_paths:

    folder = Path(checkpoint).parent

    for filename in METADATA_FILENAMES:

        metadata_path = folder / filename

        if (
            not metadata_path.is_file()
            or metadata_path in inspected
        ):
            continue

        inspected.add(metadata_path)

        try:
            with metadata_path.open(
                "r", encoding="utf-8"
            ) as f:
                info = json.load(f)

            if not isinstance(info, dict):
                continue

            metadata_found += 1

            print("\n" + "-" * 65)
            print("Metadata:", metadata_path)

            relevant = {
                key: value
                for key, value in info.items()
                if key.lower() in INTERESTING_KEYS
            }

            if relevant:
                print(json.dumps(
                    relevant, indent=2,
                    default=str
                )[:3500])
            else:
                print(
                    "Available keys:",
                    list(info.keys())[:25]
                )

        except Exception as exc:
            print(
                f"Could not read {metadata_path}: {exc}"
            )

print("\n" + "=" * 65)
print("Metadata files inspected:", metadata_found)
print("BLOCK 2.3 COMPLETE")
print("No model weights were loaded.")


CHECKPOINT METADATA INSPECTION

-----------------------------------------------------------------
Metadata: /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/block12_smoke_lambda_1/config.json
{
  "model_config": {
    "n_atom": 55,
    "n_bond": 4,
    "D": 2,
    "F_e": 16,
    "F_h": [
      32,
      64,
      128,
      128,
      256,
      256
    ],
    "F_skip": 256,
    "F_c": [
      512
    ],
    "F_policy": 128,
    "n_rnn": 3
  }
}

-----------------------------------------------------------------
Metadata: /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/config.json
{
  "model_config": {
    "n_atom": 55,
    "n_bond": 4,
    "D": 2,
    "F_e": 16,
    "F_h": [
      32,
      64,
      128,
      128,
      256,
      256
    ],
    "F_skip": 256,
    "F_c": [
      512
    ],
    "F_policy": 128,
    "n_rnn": 3
  }
}

-----------------------------------------------------------------
Metad

## Verify the First Three Checkpoints

In [9]:
import torch
import json
import pickle
import gc
from pathlib import Path
from collections import OrderedDict

In [10]:
# ------------------------------------------------------------
# A. MODEL CONFIGURATION — EDIT HERE
# ------------------------------------------------------------

PHASE_1_MODELS = OrderedDict({
    "finetuned_cdgcn_to_ctddg": {
        "directory": REWORKED_ROOT / (
            "finetuning/finetuning_runs/"
            "cdgcn_to_ctddg_d1_bs32_lambda0p1_v1"
        ),
        "checkpoint": "ckpt_best.pt",
        "conditioned": True,
    },

    "pretrained_no_toxicity": {
        "directory": REWORKED_ROOT / (
            "PRETRAINING_OUTPUT/no_toxicity/training_runs/"
            "chembl_B32_K5_alpha08_seed42"
        ),
        "checkpoint": "best.pt",
        "conditioned": False,
    },

    "pretrained_with_toxicity": {
        "directory": REWORKED_ROOT / (
            "PRETRAINING_OUTPUT/toxicity_aware/runs/"
            "ctddg_main_bs32_lambda2"
        ),
        "checkpoint": "model_best.pt",
        "conditioned": False,
    },
})

# These files must be from your own trusted training runs.
TRUST_MY_LOCAL_CHECKPOINTS = True

MODEL_CHECKPOINT_SUMMARY = {}



In [11]:

# ------------------------------------------------------------
# B. EXTRACT STATE DICTIONARY
# ------------------------------------------------------------

def extract_state_dict(checkpoint):

    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint must contain a dictionary.")

    possible_keys = [
        "model",
        "model_state_dict",
        "state_dict",
        "model_weights",
        "weights",
        "net",
    ]

    for key in possible_keys:
        candidate = checkpoint.get(key)

        if isinstance(candidate, dict):
            tensors = [
                v for v in candidate.values()
                if isinstance(v, torch.Tensor)
            ]

            if len(tensors) >= 5:
                return candidate, key

    tensors = [
        v for v in checkpoint.values()
        if isinstance(v, torch.Tensor)
    ]

    if len(tensors) >= 5:
        return checkpoint, "root"

    raise ValueError(
        "Cannot identify model state_dict in checkpoint."
    )




In [12]:
# ------------------------------------------------------------
# C. VERIFY EACH MODEL
# ------------------------------------------------------------

for model_name, info in PHASE_1_MODELS.items():

    folder = info["directory"]
    checkpoint_path = folder / info["checkpoint"]

    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    assert checkpoint_path.is_file(), checkpoint_path

    print("Checkpoint:", checkpoint_path)
    print(
        "Size (MB):",
        round(checkpoint_path.stat().st_size / 1024**2, 2)
    )

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

    except pickle.UnpicklingError:

        if not TRUST_MY_LOCAL_CHECKPOINTS:
            raise RuntimeError(
                "Checkpoint requires trusted pickle loading."
            )

        print("Using trusted local checkpoint loader.")

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

    state, state_location = extract_state_dict(checkpoint)

    print("State dictionary location:", state_location)
    print("Weight tensors:", sum(
        isinstance(v, torch.Tensor)
        for v in state.values()
    ))

    print("\nTop-level checkpoint keys:")
    print(list(checkpoint.keys())[:20])

    conditional_keys = [
        k for k in state
        if "protein_projections" in k
        or "init_head" in k
    ]

    print(
        "\nProtein-conditioning parameter keys:",
        len(conditional_keys)
    )

    if info["conditioned"] and not conditional_keys:
        raise ValueError(
            f"{model_name}: Expected conditional model "
            "parameters but found none."
        )

    print("\nExample model parameters:")

    for key in list(state.keys())[:8]:
        value = state[key]
        if isinstance(value, torch.Tensor):
            print(f"  {key}: {tuple(value.shape)}")

    


MODEL: finetuned_cdgcn_to_ctddg
Checkpoint: /home/muhamed/repo/CTDDG/REWORKED_CODE/finetuning/finetuning_runs/cdgcn_to_ctddg_d1_bs32_lambda0p1_v1/ckpt_best.pt
Size (MB): 50.01
State dictionary location: model
Weight tensors: 93

Top-level checkpoint keys:
['model', 'signature', 'step', 'epoch', 'validation_nll']

Protein-conditioning parameter keys: 14

Example model parameters:
  init_atom_logits: (43,)
  backbone.atom_embedding.weight: (43, 16)
  backbone.mask_embedding.weight: (3, 16)
  backbone.convs.0.W: (112, 32)
  backbone.convs.1.W: (224, 64)
  backbone.convs.2.W: (448, 128)
  backbone.convs.3.W: (896, 128)
  backbone.convs.4.W: (896, 256)

MODEL: pretrained_no_toxicity
Checkpoint: /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/best.pt
Size (MB): 79.93
Using trusted local checkpoint loader.
State dictionary location: model
Weight tensors: 79

Top-level checkpoint keys:
['signature', 'completed_steps', 'best_val_

In [13]:
    # --------------------------------------------------------
    # D. READ AVAILABLE MODEL METADATA
    # --------------------------------------------------------

    config = {}

    for filename in [
        "run_meta.json",
        "config.json",
        "run_config.json",
        "model_config.json",
    ]:

        path = folder / filename

        if not path.is_file():
            continue

        metadata = json.loads(path.read_text())

        if not isinstance(metadata, dict):
            continue

        signature = metadata.get("signature", {})

        candidates = [
            metadata.get("model_config"),
            signature.get("model_config")
                if isinstance(signature, dict) else None,
            metadata.get("config"),
        ]

        for candidate in candidates:
            if isinstance(candidate, dict) and (
                "n_atom" in candidate
            ):
                config = candidate
                break

        if config:
            break

    print("\nModel architecture:")
    if config:
        print(json.dumps(config, indent=2))
    else:
        print("Not found in nearby JSON metadata.")

    MODEL_CHECKPOINT_SUMMARY[model_name] = {
        "checkpoint": str(checkpoint_path),
        "conditioned": info["conditioned"],
        "state_location": state_location,
        "n_atom": config.get("n_atom"),
        "n_bond": config.get("n_bond"),
        "receptive_field": config.get("D"),
        "conditional_parameter_count": len(conditional_keys),
    }

    del checkpoint, state
    gc.collect()





Model architecture:
{
  "n_atom": 55,
  "n_bond": 4,
  "D": 2,
  "F_e": 16,
  "F_h": [
    32,
    64,
    128,
    128,
    256,
    256
  ],
  "F_skip": 256,
  "F_c": [
    512
  ],
  "F_policy": 128,
  "n_rnn": 3
}


66

In [14]:
# ------------------------------------------------------------
# E. DISPLAY SUMMARY
# ------------------------------------------------------------

import pandas as pd

print("\n" + "=" * 70)
print("PHASE 1 — CHECKPOINT SUMMARY")
print("=" * 70)

display(
    pd.DataFrame.from_dict(
        MODEL_CHECKPOINT_SUMMARY,
        orient="index"
    )
)

print("\nBLOCK 3 COMPLETE")
print("All checkpoints inspected on CPU.")
print("No molecules generated.")



PHASE 1 — CHECKPOINT SUMMARY


,checkpoint,conditioned,state_location,n_atom,n_bond,receptive_field,conditional_parameter_count
pretrained_with_toxicity,/home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/toxicity_aware/runs/ctddg_main_bs32_lambda2/model_best.pt,False,model_state_dict,55,4,2,0



BLOCK 3 COMPLETE
All checkpoints inspected on CPU.
No molecules generated.


## Check Generation Code and Atom Vocabulary

In [16]:
import json
import os
import re
import hashlib
from pathlib import Path


In [17]:

# ------------------------------------------------------------
# A. INSPECT EXISTING GENERATION IMPLEMENTATION
# ------------------------------------------------------------

notebook_path = (
    REWORKED_ROOT
    / "testing/CTDDG_Five_Model_Generation.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

patterns = [
    r"^\s*class\s+",
    r"^\s*def\s+read_checkpoint",
    r"^\s*def\s+.*sampl",
    r"\.sample\(",
    r"get_mol_spec\(",
    r"atom_types",
    r"MAX_ATTEMPTS",
    r"BATCH_SIZE",
    r"MAX_ITER",
    r"torch\.no_grad",
    r"torch\.inference_mode",
]

combined_pattern = re.compile(
    "|".join(patterns),
    re.IGNORECASE
)

print("=" * 70)
print("GENERATION IMPLEMENTATION")
print("=" * 70)

matches = []

for cell_number, cell in enumerate(
    notebook["cells"], start=1
):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", [])
    code = "".join(source) if isinstance(source, list) else source
    lines = code.splitlines()

    for index, line in enumerate(lines):

        if combined_pattern.search(line):
            matches.append({
                "cell": cell_number,
                "line": index + 1,
                "text": line.strip()[:180]
            })

print("Relevant code references:", len(matches))

for item in matches[:65]:
    print(
        f"Cell {item['cell']:>2}, "
        f"line {item['line']:>3}: {item['text']}"
    )

if len(matches) > 65:
    print(f"... {len(matches)-65} additional matches")




GENERATION IMPLEMENTATION
Relevant code references: 70
Cell  3, line  18: LEGACY_ATOM_TYPES = REWORKED / 'chembl_preprocessed/atom_types.txt'
Cell  3, line  19: CLEAN_ATOM_TYPES = REWORKED / 'PREPROCESSING_OUTPUT/vocabulary/atom_types.txt'
Cell  3, line  33: 'kind': 'legacy_unconditional', 'vocab': LEGACY_ATOM_TYPES,
Cell  3, line  40: 'kind': 'legacy_unconditional', 'vocab': LEGACY_ATOM_TYPES,
Cell  3, line  46: 'kind': 'clean_unconditional', 'vocab': CLEAN_ATOM_TYPES,
Cell  3, line  53: 'kind': 'bindingdb_conditional', 'vocab': LEGACY_ATOM_TYPES,
Cell  3, line  63: BATCH_SIZE = 8
Cell  3, line  66: MAX_ATTEMPTS_PER_RUN = 3000
Cell  3, line  72: SMOKE_MAX_ATTEMPTS = 200
Cell  3, line  80: effective_max_attempts = SMOKE_MAX_ATTEMPTS
Cell  3, line  84: effective_max_attempts = MAX_ATTEMPTS_PER_RUN
Cell  3, line  88: assert BATCH_SIZE >= 1 and MAX_ACTIONS >= 1 and MAX_ATOMS >= 1
Cell  3, line  89: assert effective_target > 0 and effective_runs > 0 and effective_max_attempts >= effective_

In [18]:
# ------------------------------------------------------------
# B. LOCATE ATOM VOCABULARY FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ATOM VOCABULARY DISCOVERY")
print("=" * 70)

SKIP_FOLDERS = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "node_modules",
    "site-packages",
    "generated_samples",
}

vocabulary_files = []

for current_dir, dirs, files in os.walk(REWORKED_ROOT):

    dirs[:] = [
        d for d in dirs
        if d not in SKIP_FOLDERS
    ]

    for filename in files:

        if filename.lower() in {
            "atom_types.txt",
            "atom_type.txt",
            "atom_vocab.txt",
        }:

            vocabulary_files.append(
                Path(current_dir) / filename
            )

vocabulary_files = sorted(set(vocabulary_files))

print("Vocabulary files found:", len(vocabulary_files))

for path in vocabulary_files:

    content = path.read_bytes()

    lines = [
        line.strip()
        for line in content.decode(
            "utf-8", errors="replace"
        ).splitlines()
        if line.strip()
    ]

    digest = hashlib.sha256(content).hexdigest()

    print("\nFile:", path)
    print("Nonempty lines:", len(lines))
    print("SHA256:", digest[:20])
    print("First 5 entries:", lines[:5])





ATOM VOCABULARY DISCOVERY
Vocabulary files found: 5

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA/atom_types.txt
Nonempty lines: 65
SHA256: a5781b7c1c32a8396292
First 5 entries: ['P,-1,0', 'Br,0,0', 'Sb,0,0', 'Cl,1,0', 'Li,0,0']

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/vocabulary/atom_types.txt
Nonempty lines: 55
SHA256: 3f2a10321e1301f7a45f
First 5 entries: ['B,-1,0', 'B,-1,3', 'B,0,0', 'Br,0,0', 'C,-1,0']

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/atom_types.txt
Nonempty lines: 43
SHA256: 98c9c49f5a4bfd8450a4
First 5 entries: ['B,0,0', 'Br,0,0', 'Br,0,1', 'C,-1,0', 'C,-1,1']

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/model_weights/chembl_B32_K5_alpha08_seed42/atom_types.txt
Nonempty lines: 55
SHA256: 3f2a10321e1301f7a45f
First 5 entries: ['B,-1,0', 'B,-1,3', 'B,0,0', 'Br,0,0', 'C,-1,0']

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/model_weights/nitesh_model/atom_types.txt
Nonempty lines: 67
SHA256: 9e45837809d98240490a
First 

In [19]:
# ------------------------------------------------------------
# C. CONFIRM GPU ENVIRONMENT
# ------------------------------------------------------------

import torch

print("\n" + "=" * 70)
print("GPU")
print("=" * 70)

assert torch.cuda.is_available(), (
    "Run on your L40 GPU allocation."
)

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

properties = torch.cuda.get_device_properties(0)

print(
    "GPU memory (GiB):",
    round(properties.total_memory / 1024**3, 2)
)

print("\nBLOCK 4 COMPLETE")
print("Generation implementation inspected.")
print("No molecules generated.")



GPU
GPU: NVIDIA L40
CUDA version: 12.1
GPU memory (GiB): 44.37

BLOCK 4 COMPLETE
Generation implementation inspected.
No molecules generated.


## Prepare the Exact Model Architecture

In [20]:

# ============================================================
# BLOCK 5 — REUSE VERIFIED CTDDG GENERATION CODE
# ============================================================

import ast
import json
import math
import random
import time
import gc
import hashlib
import pickle
import numpy as np
import pandas as pd
import scipy
import torch
import h5py

from scipy import sparse
from torch import nn
from torch.nn import functional as F
from rdkit import Chem, RDLogger
from dataclasses import dataclass
from pathlib import Path


In [21]:

# ------------------------------------------------------------
# A. GPU AND GENERATION SETTINGS
# ------------------------------------------------------------

DEVICE = torch.device("cuda:0")
assert torch.cuda.is_available()

MAX_ACTIONS = 120
MAX_ATOMS = 120
MAX_ITER_PER_MOL = 120
GREEDY = False
SEED = 42
BATCH_SIZE = 16

TRUST_LOCAL_CHECKPOINT = True

ATOM_TYPES_FILE = None
_mol_spec = None

MODEL_ROOT = REWORKED_ROOT / "model_weights"

LEGACY_ATOM_TYPES = (
    REWORKED_ROOT / "chembl_preprocessed/atom_types.txt"
)

CLEAN_ATOM_TYPES = (
    REWORKED_ROOT
    / "PREPROCESSING_OUTPUT/vocabulary/atom_types.txt"
)

# These dimensions/architectures come from training metadata.
DEFAULT_CONFIG = {
    "n_atom": 0,
    "n_bond": 4,
    "D": 2,
    "F_e": 16,
    "F_h": [32, 64, 128, 128, 256, 256],
    "F_skip": 256,
    "F_c": [512],
    "F_policy": 128,
    "n_rnn": 3,
}

CONFIG_KEYS = tuple(DEFAULT_CONFIG)

assert len(LEGACY_ATOM_TYPES.read_text().splitlines()) == 43
assert len(CLEAN_ATOM_TYPES.read_text().splitlines()) == 55



In [22]:
# ------------------------------------------------------------
# B. LOAD FUNCTION/CLASS DEFINITIONS ONLY
# ------------------------------------------------------------

with REFERENCE_NOTEBOOK.open("r", encoding="utf-8") as f:
    reference = json.load(f)

definition_markers = [
    "class MoleculeSpec",
    "class CTDDGPretrainer",
    "class _MolState",
    "def read_checkpoint(",
    "def convert_graph_for_output(",
    "def sample_batch(",
]

for marker in definition_markers:

    matching_cells = []

    for i, cell in enumerate(reference["cells"], 1):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        code = "".join(source) if isinstance(source, list) else source

        if marker in code:
            matching_cells.append((i, code))

    assert len(matching_cells) == 1, (
        f"Expected exactly one cell containing {marker}; "
        f"found {len(matching_cells)}"
    )

    cell_number, code = matching_cells[0]

    tree = ast.parse(code)

    # Import ONLY functions and classes.
    # Never execute the reference notebook's generation loop.
    definitions = [
        node for node in tree.body
        if isinstance(
            node, (ast.FunctionDef, ast.ClassDef)
        )
    ]

    safe_module = ast.Module(
        body=definitions,
        type_ignores=[]
    )

    exec(
        compile(
            ast.fix_missing_locations(safe_module),
            f"reference_cell_{cell_number}",
            "exec"
        ),
        globals()
    )

    print(
        f"Loaded reference cell {cell_number}: {marker}"
    )



Loaded reference cell 9: class MoleculeSpec
Loaded reference cell 12: class CTDDGPretrainer
Loaded reference cell 13: class _MolState
Loaded reference cell 15: def read_checkpoint(
Loaded reference cell 17: def convert_graph_for_output(
Loaded reference cell 19: def sample_batch(


In [23]:
# ------------------------------------------------------------
# C. DEFINE EXACT CHECKPOINT SOURCES
# ------------------------------------------------------------

finetune_dir = PHASE_1_MODELS[
    "finetuned_cdgcn_to_ctddg"
]["directory"]

nontoxic_dir = PHASE_1_MODELS[
    "pretrained_no_toxicity"
]["directory"]

toxicity_dir = PHASE_1_MODELS[
    "pretrained_with_toxicity"
]["directory"]

MODEL_SOURCES = {
    "finetuned_cdgcn_to_ctddg": {
        "kind": "bindingdb_conditional",
        "vocab": LEGACY_ATOM_TYPES,
        "candidates": [finetune_dir / "ckpt_best.pt"],
        "metadata_dir": finetune_dir,
        "dense_relu": False,
        "n_c": 6165,
    },

    "pretrained_no_toxicity": {
        "kind": "clean_unconditional",
        "vocab": CLEAN_ATOM_TYPES,
        "candidates": [nontoxic_dir / "best.pt"],
        "metadata_dir": nontoxic_dir,
        "dense_relu": True,
    },
}

# The toxicity-aware model uses 55 atoms, but its
# dense-layer activation has not yet been verified
# against the actual training implementation.

TOXICITY_DENSE_RELU_VERIFIED = None

if TOXICITY_DENSE_RELU_VERIFIED is not None:
    MODEL_SOURCES["pretrained_with_toxicity"] = {
        "kind": "toxicity_unconditional",
        "vocab": CLEAN_ATOM_TYPES,
        "candidates": [
            toxicity_dir / "model_best.pt"
        ],
        "metadata_dir": toxicity_dir,
        "dense_relu": TOXICITY_DENSE_RELU_VERIFIED,
    }

print("\n" + "=" * 65)
print("BLOCK 5 SUMMARY")
print("=" * 65)

print("GPU:", torch.cuda.get_device_name(0))
print("Reference architecture: LOADED")
print("Fine-tuned checkpoint: CONFIGURED")
print("No-toxicity checkpoint: CONFIGURED")
print(
    "Toxicity-aware checkpoint:",
    "PENDING ACTIVATION VERIFICATION"
)
print("No model weights loaded yet.")



BLOCK 5 SUMMARY
GPU: NVIDIA L40
Reference architecture: LOADED
Fine-tuned checkpoint: CONFIGURED
No-toxicity checkpoint: CONFIGURED
Toxicity-aware checkpoint: PENDING ACTIVATION VERIFICATION
No model weights loaded yet.


## GPU Sampling Smoke Test & Speed Benchmark

In [24]:
# ============================================================
# BLOCK 6 — L40 GENERATION SMOKE TEST
# ============================================================

import time
import gc
import h5py
import numpy as np
import pandas as pd
import torch

In [25]:
# ------------------------------------------------------------
# A. TEST SETTINGS
# ------------------------------------------------------------

SMOKE_MODELS = [
    "finetuned_cdgcn_to_ctddg",
    "pretrained_no_toxicity",
]

# Test increasingly large GPU batches.
SMOKE_BATCH_SIZES = [8, 16, 32]

# Use our verified protein embedding for conditional CTDDG.
PROTEIN_FOLD = "1002_103"
PROTEIN_ID = "protein1"

embedding_file = (
    EMBEDDINGS_ROOT
    / PROTEIN_FOLD
    / "prose_embeddings_avg_6165.h5"
)

with h5py.File(embedding_file, "r") as h5:
    protein_array = np.asarray(
        h5["p1"], dtype=np.float32
    ).copy()

assert protein_array.shape == (6165,)

protein_vector = torch.from_numpy(
    protein_array
).to(DEVICE)



In [26]:
# ------------------------------------------------------------
# B. LOAD EACH MODEL AND BENCHMARK
# ------------------------------------------------------------

benchmark_results = []

for model_name in SMOKE_MODELS:

    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    model, checkpoint_path, metadata, config = (
        read_checkpoint(model_name)
    )

    # IMPORTANT: update the global vocabulary used
    # by graph-to-SMILES conversion for this model.
    spec = set_atom_vocab(
        MODEL_SOURCES[model_name]["vocab"]
    )

    assert config["n_atom"] == spec.num_atom_types

    conditioned = bool(model.conditional)

    c = protein_vector if conditioned else None

    if conditioned:
        assert c.numel() == model.backbone.n_c

    model.eval()

    for batch_size in SMOKE_BATCH_SIZES:

        BATCH_SIZE = batch_size

        print(f"\nTesting batch size {batch_size}...")

        generator = torch.Generator(
            device=DEVICE
        )

        generator.manual_seed(
            SEED + batch_size
        )

        torch.cuda.reset_peak_memory_stats()

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():

            states = sample_batch(
                model,
                batch_size,
                generator,
                protein_vector=c
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        # Convert results to molecules.
        valid_smiles = []

        for state in states:

            smiles, error = convert_graph_for_output(
                state
            )

            if (
                state.status == "ended"
                and smiles is not None
            ):
                valid_smiles.append(smiles)

        valid_count = len(valid_smiles)
        unique_count = len(set(valid_smiles))

        attempts_per_second = (
            batch_size / elapsed
        )

        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        benchmark_results.append({
            "model": model_name,
            "batch_size": batch_size,
            "seconds": round(elapsed, 2),
            "attempts_per_second": round(
                attempts_per_second, 3
            ),
            "valid_molecules": valid_count,
            "unique_valid": unique_count,
            "peak_gpu_memory_GB": round(
                peak_memory_gb, 3
            ),
        })

        print(
            f"Time: {elapsed:.2f}s | "
            f"Valid: {valid_count}/{batch_size} | "
            f"Throughput: {attempts_per_second:.2f} attempts/s | "
            f"GPU memory: {peak_memory_gb:.2f} GB"
        )

    del model
    gc.collect()
    torch.cuda.empty_cache()





MODEL: finetuned_cdgcn_to_ctddg

Loading finetuned_cdgcn_to_ctddg : /home/muhamed/repo/CTDDG/REWORKED_CODE/finetuning/finetuning_runs/cdgcn_to_ctddg_d1_bs32_lambda0p1_v1/ckpt_best.pt
PASS strict load | protein-conditioned: True | atom types: 43 | weight SHA256: 33bb7063c02ed420

Testing batch size 8...
Time: 1.27s | Valid: 8/8 | Throughput: 6.31 attempts/s | GPU memory: 0.08 GB

Testing batch size 16...
Time: 0.24s | Valid: 16/16 | Throughput: 66.21 attempts/s | GPU memory: 0.08 GB

Testing batch size 32...


[22:33:44] Can't kekulize mol.  Unkekulized atoms: 2 3 4 21 22 23 24 26 28
[22:33:44] Can't kekulize mol.  Unkekulized atoms: 2 3 4 21 22 23 24 26 28


Time: 0.37s | Valid: 31/32 | Throughput: 86.51 attempts/s | GPU memory: 0.08 GB

MODEL: pretrained_no_toxicity

Loading pretrained_no_toxicity : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/best.pt
Trusted local checkpoint requires weights_only=False: UnpicklingError
PASS strict load | protein-conditioned: False | atom types: 55 | weight SHA256: 179167eb22a363ae

Testing batch size 8...
Time: 0.14s | Valid: 8/8 | Throughput: 56.76 attempts/s | GPU memory: 0.06 GB

Testing batch size 16...
Time: 0.22s | Valid: 16/16 | Throughput: 73.05 attempts/s | GPU memory: 0.06 GB

Testing batch size 32...
Time: 0.27s | Valid: 31/32 | Throughput: 119.19 attempts/s | GPU memory: 0.06 GB


[22:33:45] Can't kekulize mol.  Unkekulized atoms: 4 5 10 11 12
[22:33:45] Can't kekulize mol.  Unkekulized atoms: 4 5 10 11 12


In [27]:
# ------------------------------------------------------------
# C. DISPLAY COMPARISON
# ------------------------------------------------------------

BENCHMARK_DF = pd.DataFrame(benchmark_results)

print("\n" + "=" * 70)
print("L40 GPU — SAMPLING PERFORMANCE")
print("=" * 70)

display(BENCHMARK_DF)

print("\nFastest tested batch size per model:")

for model_name in SMOKE_MODELS:

    subset = BENCHMARK_DF[
        BENCHMARK_DF["model"] == model_name
    ]

    best = subset.loc[
        subset["attempts_per_second"].idxmax()
    ]

    print(
        model_name,
        "| batch size:", int(best["batch_size"]),
        "| attempts/s:", best["attempts_per_second"]
    )

print("\nBLOCK 6 SMOKE TEST COMPLETE")
print("No production samples saved.")



L40 GPU — SAMPLING PERFORMANCE


,model,batch_size,seconds,attempts_per_second,valid_molecules,unique_valid,peak_gpu_memory_GB
0,finetuned_cdgcn_to_ctddg,8,1.27,6.314,8,8,0.080
1,finetuned_cdgcn_to_ctddg,16,0.24,66.212,16,16,0.081
2,finetuned_cdgcn_to_ctddg,32,0.37,86.513,31,31,0.084
3,pretrained_no_toxicity,8,0.14,56.758,8,8,0.057
4,pretrained_no_toxicity,16,0.22,73.054,16,16,0.058
5,pretrained_no_toxicity,32,0.27,119.195,31,31,0.059



Fastest tested batch size per model:
finetuned_cdgcn_to_ctddg | batch size: 32 | attempts/s: 86.513
pretrained_no_toxicity | batch size: 32 | attempts/s: 119.195

BLOCK 6 SMOKE TEST COMPLETE
No production samples saved.


## Optimize L40 Sampling and Inspect the Third Model

In [28]:

# ============================================================
# BLOCK 7.1 — VERIFY TOXICITY MODEL IMPLEMENTATION
# ============================================================

import json
import re
import torch
from pathlib import Path

TOXICITY_DENSE_RELU_VERIFIED = None

tox_checkpoint = (
    PHASE_1_MODELS["pretrained_with_toxicity"]["directory"]
    / "model_best.pt"
)

obj = torch.load(
    tox_checkpoint,
    map_location="cpu",
    weights_only=True
)

cfg = obj.get("config", {})

print("=" * 65)
print("TOXICITY-AWARE MODEL CONFIGURATION")
print("=" * 65)

print(json.dumps(cfg, indent=2, default=str)[:4000])

# Accept only an explicit Boolean setting.
for candidate in [
    cfg,
    cfg.get("model_config", {}),
]:
    if isinstance(candidate, dict):
        value = candidate.get("dense_relu")
        if type(value) is bool:
            TOXICITY_DENSE_RELU_VERIFIED = value

# Find relevant training implementations for inspection.
SEARCH_DIRS = [
    REWORKED_ROOT / "PRETRAINING_OUTPUT/toxicity_aware",
    REWORKED_ROOT / "pretraining",
]

print("\nTRAINING CODE REFERENCES")

matches = []

for root in SEARCH_DIRS:
    if not root.is_dir():
        continue

    for path in root.rglob("*"):

        if path.suffix not in {".py", ".ipynb"}:
            continue
        if path.stat().st_size > 15 * 1024**2:
            continue

        try:
            if path.suffix == ".ipynb":
                nb = json.loads(path.read_text())
                lines = []
                for cell in nb.get("cells", []):
                    if cell.get("cell_type") == "code":
                        src = cell.get("source", [])
                        text = "".join(src) if isinstance(src, list) else src
                        lines.extend(text.splitlines())
            else:
                lines = path.read_text(
                    errors="replace"
                ).splitlines()

            for i, line in enumerate(lines):
                if re.search(
                    r"dense_relu|for\s+\w+\s+in\s+self\.dense|"
                    r"class GraphBackbone",
                    line,
                    re.I
                ):
                    matches.append((
                        path,
                        i + 1,
                        "\n".join(lines[
                            max(0, i - 2):min(len(lines), i + 7)
                        ])
                    ))

        except Exception:
            continue

for path, line_number, snippet in matches[:18]:
    print(f"\n{path.name}, line {line_number}")
    print(snippet[:850])

print("\nExplicit checkpoint activation:",
      TOXICITY_DENSE_RELU_VERIFIED)

if TOXICITY_DENSE_RELU_VERIFIED is None:
    print(
        "REVIEW REQUIRED: inspect the training implementation "
        "before enabling toxicity-aware generation."
    )
else:
    print("Explicit dense activation configuration found.")

del obj


TOXICITY-AWARE MODEL CONFIGURATION
{
  "model_config": {
    "n_atom": 55,
    "n_bond": 4,
    "D": 2,
    "F_e": 16,
    "F_h": [
      32,
      64,
      128,
      128,
      256,
      256
    ],
    "F_skip": 256,
    "F_c": [
      512
    ],
    "F_policy": 128,
    "n_rnn": 3
  },
  "seed": 42,
  "tox_lambda": 2.0,
  "learning_rate": 0.001,
  "weight_decay": 0.0,
  "batch_size": 32,
  "k_paths": 5,
  "alpha": 0.8,
  "grad_clip": 3.0,
  "lr_decay": 0.001,
  "lr_decay_step": 100,
  "train_count": 1035981,
  "val_count": 512,
  "train_indices_sha256": "ab1a29359121b39fe167bbdb56347c06a2a5058d0eaa2675370bda269f2f5830",
  "val_indices_sha256": "f973befa208ce437f0bd6a864f0a86e90dd12a8bada98f7459359cffac8396e8",
  "chembl_sha256": "79920905a6d7784383dc6a944760b6e3290f1ae49ec89a3d1b8cfa6dbc2a67d4",
  "atom_vocab_sha256": "3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f"
}

TRAINING CODE REFERENCES

CTDDG_pretraining_final copy.ipynb, line 518
        return torch.cat

In [29]:

# ============================================================
# BLOCK 7.2 — L40 BATCH-SIZE OPTIMIZATION
# ============================================================

import time
import gc
import numpy as np
import pandas as pd
import torch

TEST_BATCH_SIZES = [32, 64, 128]
REPEATS_PER_SIZE = 3

OPTIMAL_BATCH_SIZES = {}
performance_rows = []

for model_name in [
    "finetuned_cdgcn_to_ctddg",
    "pretrained_no_toxicity",
]:

    model, _, _, config = read_checkpoint(model_name)

    # Global vocabulary used by graph-to-SMILES conversion.
    spec = set_atom_vocab(
        MODEL_SOURCES[model_name]["vocab"]
    )

    c = protein_vector if model.conditional else None

    for batch_size in TEST_BATCH_SIZES:

        speeds = []
        valid_counts = []
        memory_gb = []

        for repeat in range(REPEATS_PER_SIZE):

            generator = torch.Generator(device=DEVICE)
            generator.manual_seed(
                SEED + batch_size + repeat * 1000
            )

            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

            start = time.perf_counter()

            with torch.inference_mode():
                states = sample_batch(
                    model,
                    batch_size,
                    generator,
                    protein_vector=c
                )

            torch.cuda.synchronize()

            # Include SMILES conversion in timing.
            valid = 0

            for state in states:
                smiles, _ = convert_graph_for_output(state)
                if state.status == "ended" and smiles is not None:
                    valid += 1

            elapsed = time.perf_counter() - start

            speeds.append(batch_size / elapsed)
            valid_counts.append(valid)

            memory_gb.append(
                torch.cuda.max_memory_allocated() / 1024**3
            )

        performance_rows.append({
            "model": model_name,
            "batch_size": batch_size,
            "median_attempts_per_sec": round(
                float(np.median(speeds)), 2
            ),
            "mean_valid_fraction": round(
                float(np.mean(valid_counts)) / batch_size, 3
            ),
            "peak_memory_GB": round(max(memory_gb), 3),
        })

    del model
    gc.collect()
    torch.cuda.empty_cache()

PERFORMANCE_DF = pd.DataFrame(performance_rows)

for name, group in PERFORMANCE_DF.groupby("model"):
    best = group.loc[
        group["median_attempts_per_sec"].idxmax()
    ]

    OPTIMAL_BATCH_SIZES[name] = int(best["batch_size"])

print("=" * 65)
print("L40 BATCH OPTIMIZATION RESULTS")
print("=" * 65)

display(PERFORMANCE_DF)

print("\nSelected batch sizes:")
print(OPTIMAL_BATCH_SIZES)

print("\nBLOCK 7 COMPLETE")



Loading finetuned_cdgcn_to_ctddg : /home/muhamed/repo/CTDDG/REWORKED_CODE/finetuning/finetuning_runs/cdgcn_to_ctddg_d1_bs32_lambda0p1_v1/ckpt_best.pt
PASS strict load | protein-conditioned: True | atom types: 43 | weight SHA256: 33bb7063c02ed420


[22:40:23] Can't kekulize mol.  Unkekulized atoms: 2 3 4 21 22 23 24 26 28
[22:40:23] Can't kekulize mol.  Unkekulized atoms: 2 3 4 21 22 23 24 26 28
[22:40:24] non-ring atom 1 marked aromatic
[22:40:24] non-ring atom 1 marked aromatic
[22:40:24] Can't kekulize mol.  Unkekulized atoms: 28 29 30
[22:40:24] Can't kekulize mol.  Unkekulized atoms: 28 29 30
[22:40:24] Can't kekulize mol.  Unkekulized atoms: 32 33 34 35 36 39
[22:40:24] Can't kekulize mol.  Unkekulized atoms: 32 33 34 35 36 39
[22:40:25] Explicit valence for atom # 13 C, 7, is greater than permitted
[22:40:25] Can't kekulize mol.  Unkekulized atoms: 12 13 14 15 17 25 28
[22:40:25] Can't kekulize mol.  Unkekulized atoms: 12 13 14 15 17 25 28
[22:40:25] Can't kekulize mol.  Unkekulized atoms: 5 6 7 11 12 13 14
[22:40:25] Can't kekulize mol.  Unkekulized atoms: 5 6 7 11 12 13 14
[22:40:26] non-ring atom 10 marked aromatic
[22:40:26] non-ring atom 10 marked aromatic
[22:40:26] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 1


Loading pretrained_no_toxicity : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/training_runs/chembl_B32_K5_alpha08_seed42/best.pt
Trusted local checkpoint requires weights_only=False: UnpicklingError
PASS strict load | protein-conditioned: False | atom types: 55 | weight SHA256: 179167eb22a363ae


[22:40:30] Can't kekulize mol.  Unkekulized atoms: 4 5 10 11 12
[22:40:30] Can't kekulize mol.  Unkekulized atoms: 4 5 10 11 12
[22:40:30] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 7 22 23 24 25 26 27
[22:40:30] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 7 22 23 24 25 26 27
[22:40:31] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 22 23 24
[22:40:31] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 22 23 24
[22:40:31] Can't kekulize mol.  Unkekulized atoms: 14 15 16 17 18 20 21
[22:40:31] Can't kekulize mol.  Unkekulized atoms: 14 15 16 17 18 20 21
[22:40:32] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 13 21
[22:40:32] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 13 21
[22:40:32] Explicit valence for atom # 0 C, 5, is greater than permitted
[22:40:33] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 22 23 24 25 26
[22:40:33] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 22 23 24 25 26


L40 BATCH OPTIMIZATION RESULTS


,model,batch_size,median_attempts_per_sec,mean_valid_fraction,peak_memory_GB
0,finetuned_cdgcn_to_ctddg,32,82.03,0.958,0.084
1,finetuned_cdgcn_to_ctddg,64,104.91,0.974,0.091
2,finetuned_cdgcn_to_ctddg,128,118.67,0.971,0.124
3,pretrained_no_toxicity,32,117.04,0.979,0.059
4,pretrained_no_toxicity,64,140.16,0.990,0.064
5,pretrained_no_toxicity,128,158.33,0.992,0.076



Selected batch sizes:
{'finetuned_cdgcn_to_ctddg': 128, 'pretrained_no_toxicity': 128}

BLOCK 7 COMPLETE


## Resumable Sample Generation

In [30]:

# ============================================================
# BLOCK 8.1 — GENERATION SETTINGS
# ============================================================

import json
import hashlib
import h5py
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# EDITABLE SETTINGS
# ------------------------------------------------------------

SMOKE_TEST = True

FOLDS_TO_SAMPLE = [
    "1002_103",
    "1042_104",
]

TOTAL_RUNS = 10
TARGET_PER_RUN = 100
MAX_ATTEMPTS_PER_RUN = 3000

SMOKE_PROTEINS_PER_FOLD = 1
SMOKE_RUNS = 1
SMOKE_TARGET = 10
SMOKE_MAX_ATTEMPTS = 200

SAVE_RAW_EVERY_BATCHES = 20
SHOW_PROGRESS_EVERY_BATCHES = 25

if SMOKE_TEST:
    effective_runs = SMOKE_RUNS
    effective_target = SMOKE_TARGET
    effective_max_attempts = SMOKE_MAX_ATTEMPTS
    SAVE_ROOT = SAMPLES_ROOT / "_smoke_phase1_v1"
else:
    effective_runs = TOTAL_RUNS
    effective_target = TARGET_PER_RUN
    effective_max_attempts = MAX_ATTEMPTS_PER_RUN
    SAVE_ROOT = SAMPLES_ROOT / "phase1_v1"

# ------------------------------------------------------------
# VERIFY THIRD MODEL BEFORE INCLUDING IT
# ------------------------------------------------------------

if type(TOXICITY_DENSE_RELU_VERIFIED) is bool:

    MODEL_SOURCES["pretrained_with_toxicity"] = {
        "kind": "toxicity_unconditional",
        "vocab": CLEAN_ATOM_TYPES,
        "candidates": [
            PHASE_1_MODELS[
                "pretrained_with_toxicity"
            ]["directory"] / "model_best.pt"
        ],
        "metadata_dir": PHASE_1_MODELS[
            "pretrained_with_toxicity"
        ]["directory"],
        "dense_relu": TOXICITY_DENSE_RELU_VERIFIED,
    }

MODEL_ORDER = [
    "finetuned_cdgcn_to_ctddg",
    "pretrained_no_toxicity",
]

if "pretrained_with_toxicity" in MODEL_SOURCES:
    MODEL_ORDER.append("pretrained_with_toxicity")
else:
    print(
        "NOTE: Third model not enabled yet. "
        "Training activation still needs verification."
    )

OPTIMAL_BATCH_SIZES.setdefault(
    "pretrained_with_toxicity", 32
)

# ------------------------------------------------------------
# LOAD VERIFIED PROTEIN MAPPINGS
# ------------------------------------------------------------

FOLD_MANIFESTS = {}
FOLD_VECTORS = {}

for fold in FOLDS_TO_SAMPLE:

    folder = EMBEDDINGS_ROOT / fold
    manifest = pd.read_csv(
        folder / "protein_manifest.csv"
    )

    vectors = {}

    with h5py.File(
        folder / "prose_embeddings_avg_6165.h5", "r"
    ) as h5:

        for row in manifest.itertuples(index=False):

            index = int(row.cleaned_order_id)
            expected_id = f"protein{index}"

            assert row.protein_id == expected_id

            vec = np.asarray(
                h5[f"p{index}"], dtype=np.float32
            )

            assert vec.shape == (6165,)
            assert np.isfinite(vec).all()

            vectors[row.protein_id] = vec

    FOLD_MANIFESTS[fold] = manifest
    FOLD_VECTORS[fold] = vectors

print("=" * 65)
print("GENERATION CONFIGURATION")
print("=" * 65)

print("Mode:", "SMOKE TEST" if SMOKE_TEST else "FULL")
print("Models:", MODEL_ORDER)
print("Folds:", FOLDS_TO_SAMPLE)
print("Runs per slot:", effective_runs)
print("Unique molecules per run:", effective_target)
print("Maximum attempts per run:", effective_max_attempts)
print("Output:", SAVE_ROOT)


NOTE: Third model not enabled yet. Training activation still needs verification.
GENERATION CONFIGURATION
Mode: SMOKE TEST
Models: ['finetuned_cdgcn_to_ctddg', 'pretrained_no_toxicity']
Folds: ['1002_103', '1042_104']
Runs per slot: 1
Unique molecules per run: 10
Maximum attempts per run: 200
Output: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/generated_samples/_smoke_phase1_v1


In [36]:

# ============================================================
# BLOCK 8.2 — GENERATE & SAVE MOLECULE SAMPLES
# ============================================================

import os
import json
import gc
import contextlib
import time
from collections import Counter

# ------------------------------------------------------------
# HELPER: ATOMIC JSON SAVE
# ------------------------------------------------------------

def save_json_atomic(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(
        data, indent=2, default=str
    ))
    os.replace(temporary, path)


def verify_saved_run(run_dir, expected, target):

    metadata_path = run_dir / "run_info.json"

    if not metadata_path.exists():
        if run_dir.exists() and any(run_dir.iterdir()):
            raise FileExistsError(
                f"Incomplete run requires review: {run_dir}"
            )
        return None

    meta = json.loads(metadata_path.read_text())

    for key, value in expected.items():
        if meta.get(key) != value:
            raise ValueError(
                f"Saved run configuration mismatch: "
                f"{run_dir} [{key}]"
            )

    accepted_path = run_dir / "accepted.csv"
    raw_path = run_dir / "raw_attempts.csv"

    assert accepted_path.is_file()
    assert raw_path.is_file()

    assert file_hash(accepted_path) == meta["accepted_sha256"]
    assert file_hash(raw_path) == meta["raw_sha256"]

    accepted = pd.read_csv(accepted_path)
    raw = pd.read_csv(raw_path)

    assert len(accepted) == target
    assert accepted.SMILES.nunique() == target
    assert len(raw) == meta["attempts"]

    flags = raw.saved_in_main_csv.astype(str).str.lower().isin(
        ("true", "1")
    )

    assert Counter(raw.loc[flags, "SMILES"]) == Counter(
        accepted.SMILES
    )

    return accepted, raw


# ------------------------------------------------------------
# GENERATION: ONE MODEL AT A TIME
# ------------------------------------------------------------

SAVE_ROOT.mkdir(parents=True, exist_ok=True)

GENERATION_SUMMARY = []

for model_name in MODEL_ORDER:

    print("\n" + "=" * 70, flush=True)
    print("MODEL:", model_name, flush=True)
    print("=" * 70, flush=True)

    model, checkpoint_path, checkpoint_meta, config = (
        read_checkpoint(model_name)
    )

    spec = set_atom_vocab(
        MODEL_SOURCES[model_name]["vocab"]
    )

    conditioned = bool(model.conditional)

    weight_hash = file_hash(checkpoint_path)
    vocab_hash = file_hash(
        MODEL_SOURCES[model_name]["vocab"]
    )

    BATCH_SIZE = OPTIMAL_BATCH_SIZES.get(model_name, 32)

    print("Batch size:", BATCH_SIZE, flush=True)
    print("Protein-conditioned:", conditioned, flush=True)

    for fold in FOLDS_TO_SAMPLE:

        manifest = FOLD_MANIFESTS[fold]

        if SMOKE_TEST:
            manifest = manifest.head(
                SMOKE_PROTEINS_PER_FOLD
            )

        for row in manifest.itertuples(index=False):

            protein_id = row.protein_id

            folder = (
                SAVE_ROOT / model_name / fold / protein_id
            )
            runs_dir = folder / "runs"

            # Protein IDs are fold-specific.
            # For unconditional models, these are sampling
            # groups only, NOT biological conditioning.
            vector = None

            if conditioned:
                vector = torch.as_tensor(
                    FOLD_VECTORS[fold][protein_id],
                    dtype=torch.float32,
                    device=DEVICE
                )

            vector_hash = (
                hashlib.sha256(
                    FOLD_VECTORS[fold][protein_id].tobytes()
                ).hexdigest()
                if conditioned else None
            )

            accepted_parts = []
            raw_parts = []

            for run_id in range(1, effective_runs + 1):

                run_dir = runs_dir / f"run_{run_id:02d}"

                # Include fold in RNG identity to avoid
                # identical random streams across folds.
                seed_identity = f"{fold}:{protein_id}"

                seed = stable_seed(
                    SEED, model_name, seed_identity, run_id
                )

                expected = {
                    "model": model_name,
                    "fold": fold,
                    "protein_id": protein_id,
                    "run_id": run_id,
                    "seed": seed,
                    "checkpoint_sha256": weight_hash,
                    "vocabulary_sha256": vocab_hash,
                    "sequence_sha256": row.sequence_sha256,
                    "conditioned_on_protein": conditioned,
                    "embedding_sha256": vector_hash,
                    "batch_size": BATCH_SIZE,
                    "target": effective_target,
                    "max_attempts": effective_max_attempts,
                    "max_actions": MAX_ACTIONS,
                    "max_atoms": MAX_ATOMS,
                    "greedy": GREEDY,
                }

                previous = verify_saved_run(
                    run_dir, expected, effective_target
                )

                if previous is not None:
                    accepted, raw = previous

                else:
                    run_dir.mkdir(
                        parents=True, exist_ok=True
                    )

                    progress_accepted = (
                        run_dir / "_progress_accepted.csv"
                    )
                    progress_raw = (
                        run_dir / "_progress_raw.csv"
                    )

                    # Store verbose decoder output in a file
                    # rather than flooding the notebook.
                    with (
                        run_dir / "generation.log"
                    ).open("w") as log:

                        with contextlib.redirect_stdout(log):

                            accepted, raw = collect_unique_samples(
                                model,
                                model_name,
                                seed_identity,
                                run_id,
                                target=effective_target,
                                max_attempts=effective_max_attempts,
                                protein_vector=vector,
                                raw_path=progress_raw,
                                accepted_path=progress_accepted,
                            )

                    if len(accepted) != effective_target:
                        raise RuntimeError(
                            f"{model_name}/{fold}/{protein_id}: "
                            f"only {len(accepted)}/{effective_target} "
                            "molecules generated. Partial logs saved."
                        )

                    accepted_path = run_dir / "accepted.csv"
                    raw_path = run_dir / "raw_attempts.csv"

                    atomic_csv(accepted, accepted_path)
                    atomic_csv(raw, raw_path)

                    meta = {
                        **expected,
                        "accepted": len(accepted),
                        "attempts": len(raw),
                        "accepted_sha256": file_hash(
                            accepted_path
                        ),
                        "raw_sha256": file_hash(raw_path),
                    }

                    save_json_atomic(
                        meta, run_dir / "run_info.json"
                    )

                    progress_accepted.unlink(missing_ok=True)
                    progress_raw.unlink(missing_ok=True)

                accepted_parts.append(accepted)
                raw_parts.append(raw)

            # Combine completed runs for evaluation.
            combined = pd.concat(
                accepted_parts, ignore_index=True
            )

            raw_combined = pd.concat(
                raw_parts, ignore_index=True
            )

            folder.mkdir(parents=True, exist_ok=True)

            main_file = (
                folder / f"{protein_id}_all_samples.csv"
            )
            raw_file = (
                folder / f"{protein_id}_raw_attempts.csv"
            )

            atomic_csv(combined, main_file)
            atomic_csv(raw_combined, raw_file)

            save_json_atomic({
                "model": model_name,
                "fold": fold,
                "protein_id": protein_id,
                "conditioned_on_protein": conditioned,
                "checkpoint": str(checkpoint_path),
                "checkpoint_sha256": weight_hash,
                "vocabulary_sha256": vocab_hash,
                "protein_sequence_sha256": row.sequence_sha256,
                "protein_embedding_sha256": vector_hash,
                "runs": effective_runs,
                "target_per_run": effective_target,
                "accepted_molecules": len(combined),
                "raw_attempts": len(raw_combined),
                "batch_size": BATCH_SIZE,
                "samples_sha256": file_hash(main_file),
                "raw_attempts_sha256": file_hash(raw_file),
            }, folder / "generation_info.json")

            GENERATION_SUMMARY.append({
                "model": model_name,
                "fold": fold,
                "protein": protein_id,
                "accepted": len(combined),
                "attempts": len(raw_combined),
            })

            print(
                f"{fold}/{protein_id}: "
                f"{len(combined)} accepted / "
                f"{len(raw_combined)} attempts",
                flush=True
            )

    del model
    gc.collect()
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATION SUMMARY")
print("=" * 70)

display(pd.DataFrame(GENERATION_SUMMARY))

print("\nResults saved under:", SAVE_ROOT)
print("BLOCK 8 COMPLETE")



MODEL: finetuned_cdgcn_to_ctddg

Loading finetuned_cdgcn_to_ctddg : /home/muhamed/repo/CTDDG/REWORKED_CODE/finetuning/finetuning_runs/cdgcn_to_ctddg_d1_bs32_lambda0p1_v1/ckpt_best.pt
PASS strict load | protein-conditioned: True | atom types: 43 | weight SHA256: 33bb7063c02ed420
Batch size: 128
Protein-conditioned: True
1002_103/protein1: 1000 accepted / 1280 attempts
1002_103/protein2: 1000 accepted / 1280 attempts
1002_103/protein3: 1000 accepted / 1280 attempts
1002_103/protein4: 1000 accepted / 1280 attempts
1002_103/protein5: 1000 accepted / 1280 attempts
1002_103/protein6: 1000 accepted / 1280 attempts
1002_103/protein7: 1000 accepted / 1280 attempts
1002_103/protein8: 1000 accepted / 1280 attempts
1002_103/protein9: 1000 accepted / 1280 attempts
1002_103/protein10: 1000 accepted / 1280 attempts
1002_103/protein11: 1000 accepted / 1280 attempts
1002_103/protein12: 1000 accepted / 1280 attempts
1002_103/protein13: 1000 accepted / 1280 attempts
1002_103/protein14: 1000 accepted / 1

,model,fold,protein,accepted,attempts
0,finetuned_cdgcn_to_ctddg,1002_103,protein1,1000,1280
1,finetuned_cdgcn_to_ctddg,1002_103,protein2,1000,1280
2,finetuned_cdgcn_to_ctddg,1002_103,protein3,1000,1280
3,finetuned_cdgcn_to_ctddg,1002_103,protein4,1000,1280
4,finetuned_cdgcn_to_ctddg,1002_103,protein5,1000,1280
...,...,...,...,...,...
601,pretrained_with_toxicity,1042_104,protein95,1000,2560
602,pretrained_with_toxicity,1042_104,protein96,1000,2560
603,pretrained_with_toxicity,1042_104,protein97,1000,2560
604,pretrained_with_toxicity,1042_104,protein98,1000,2560



Results saved under: /home/muhamed/repo/CTDDG/REWORKED_CODE/EVALUATION_OUTPUT/generated_samples/phase1_v1
BLOCK 8 COMPLETE


## Final Sample Verification

In [37]:

# ============================================================
# BLOCK 9 — FINAL GENERATION OUTPUT VERIFICATION
# Run AFTER Block 8.2 finishes
# ============================================================

import json
import pandas as pd
from pathlib import Path
from collections import Counter

assert not SMOKE_TEST
assert effective_runs == 10
assert effective_target == 100
assert SAVE_ROOT.name == "phase1_v1"

results = []
errors = []

for model_name in MODEL_ORDER:

    source = MODEL_SOURCES[model_name]
    checkpoint_sha = file_hash(source["candidates"][0])
    vocab_sha = file_hash(source["vocab"])

    conditioned = (
        source["kind"] == "bindingdb_conditional"
    )

    print(f"\nChecking {model_name}...", flush=True)

    for fold in FOLDS_TO_SAMPLE:

        manifest = FOLD_MANIFESTS[fold]

        for row in manifest.itertuples(index=False):

            protein_id = row.protein_id

            folder = (
                SAVE_ROOT / model_name / fold / protein_id
            )

            main_path = (
                folder / f"{protein_id}_all_samples.csv"
            )
            raw_path = (
                folder / f"{protein_id}_raw_attempts.csv"
            )
            info_path = folder / "generation_info.json"

            try:
                assert main_path.is_file()
                assert raw_path.is_file()
                assert info_path.is_file()

                info = json.loads(info_path.read_text())

                assert info["model"] == model_name
                assert info["fold"] == fold
                assert info["protein_id"] == protein_id
                assert info["conditioned_on_protein"] == conditioned
                assert info["checkpoint_sha256"] == checkpoint_sha
                assert info["vocabulary_sha256"] == vocab_sha
                assert info["protein_sequence_sha256"] == row.sequence_sha256

                expected_emb_sha = (
                    hashlib.sha256(
                        FOLD_VECTORS[fold][protein_id].tobytes()
                    ).hexdigest()
                    if conditioned else None
                )

                assert (
                    info["protein_embedding_sha256"]
                    == expected_emb_sha
                )

                assert info["runs"] == 10
                assert info["target_per_run"] == 100

                assert file_hash(main_path) == info["samples_sha256"]
                assert file_hash(raw_path) == info["raw_attempts_sha256"]

                accepted = pd.read_csv(
                    main_path, keep_default_na=False
                )
                raw = pd.read_csv(
                    raw_path, keep_default_na=False
                )

                assert list(accepted.columns) == [
                    "Run_ID", "SMILES"
                ]

                assert len(accepted) == 1000
                assert len(raw) == info["raw_attempts"]

                expected_counts = {
                    i: 100 for i in range(1, 11)
                }

                assert (
                    accepted.groupby("Run_ID").size().to_dict()
                    == expected_counts
                )

                flags = (
                    raw["saved_in_main_csv"]
                    .astype(str)
                    .str.lower()
                    .isin(("true", "1"))
                )

                assert Counter(
                    zip(
                        raw.loc[flags, "Run_ID"],
                        raw.loc[flags, "SMILES"]
                    )
                ) == Counter(
                    zip(
                        accepted["Run_ID"],
                        accepted["SMILES"]
                    )
                )

                for run_id in range(1, 11):

                    run_samples = accepted[
                        accepted["Run_ID"] == run_id
                    ]

                    assert run_samples.SMILES.nunique() == 100

                    run_info_path = (
                        folder / "runs"
                        / f"run_{run_id:02d}"
                        / "run_info.json"
                    )

                    assert run_info_path.is_file()

                    run_info = json.loads(
                        run_info_path.read_text()
                    )

                    assert run_info["model"] == model_name
                    assert run_info["fold"] == fold
                    assert run_info["protein_id"] == protein_id
                    assert run_info["run_id"] == run_id
                    assert run_info["target"] == 100
                    assert (
                        run_info["checkpoint_sha256"]
                        == checkpoint_sha
                    )

                results.append({
                    "model": model_name,
                    "fold": fold,
                    "protein": protein_id,
                    "runs": 10,
                    "accepted": len(accepted),
                    "attempts": len(raw),
                    "status": "PASS"
                })

            except Exception as exc:

                errors.append({
                    "model": model_name,
                    "fold": fold,
                    "protein": protein_id,
                    "error": f"{type(exc).__name__}: {exc}"
                })

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

qa_df = pd.DataFrame(
    results,
    columns=[
        "model", "fold", "protein", "runs",
        "accepted", "attempts", "status"
    ]
)

expected_slots = sum(
    len(FOLD_MANIFESTS[fold])
    for fold in FOLDS_TO_SAMPLE
) * len(MODEL_ORDER)

print("\n" + "=" * 70)
print("FINAL GENERATION VERIFICATION")
print("=" * 70)

print("Expected model/protein slots:", expected_slots)
print("Verified slots:", len(results))
print("Failed or missing slots:", len(errors))

if not qa_df.empty:
    display(
        qa_df.groupby(
            ["model", "fold"], as_index=False
        ).agg(
            proteins=("protein", "count"),
            accepted=("accepted", "sum"),
            raw_attempts=("attempts", "sum")
        )
    )

print("\nTotal accepted:", int(qa_df["accepted"].sum()))
print("Expected accepted:", expected_slots * 1000)

if errors:
    print("\nPROBLEMS FOUND:")
    display(pd.DataFrame(errors).head(20))
else:
    assert len(results) == expected_slots
    assert qa_df["accepted"].sum() == 606000

    print("\nALL CHECKS PASSED!")
    print("NOTEBOOK 2 — GENERATE SAMPLES COMPLETE")



Checking finetuned_cdgcn_to_ctddg...



Checking pretrained_no_toxicity...

Checking pretrained_with_toxicity...

FINAL GENERATION VERIFICATION
Expected model/protein slots: 606
Verified slots: 606
Failed or missing slots: 0


,model,fold,proteins,accepted,raw_attempts
0,finetuned_cdgcn_to_ctddg,1002_103,103,103000,131840
1,finetuned_cdgcn_to_ctddg,1042_104,99,99000,126720
2,pretrained_no_toxicity,1002_103,103,103000,131840
3,pretrained_no_toxicity,1042_104,99,99000,126720
4,pretrained_with_toxicity,1002_103,103,103000,263680
5,pretrained_with_toxicity,1042_104,99,99000,253440



Total accepted: 606000
Expected accepted: 606000

ALL CHECKS PASSED!
NOTEBOOK 2 — GENERATE SAMPLES COMPLETE
